In [1]:
import requests
import pandas as pd
import sys
import os
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)
from src.keywords import KEYWORDS

In [2]:
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

API_KEY = os.getenv("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY not found in .env")

SEARCH_URL = "https://www.googleapis.com/youtube/v3/search"
VIDEOS_URL = "https://www.googleapis.com/youtube/v3/videos"
CHANNELS_URL = "https://www.googleapis.com/youtube/v3/channels"

In [3]:
keywords = [
    "python",
    "machine learning",
    "data science",
    "artificial intelligence",
    "deep learning"
]

In [4]:
def search_videos(keyword, max_results=5):
    """
    Search YouTube videos using a keyword.

    Parameters:
        keyword (str): Search term.
        max_results (int): Number of videos to retrieve.

    Returns:
        list: List of YouTube video IDs.
    """
    params = {
        "part": "snippet",
        "q": keyword,
        "type": "video",
        "maxResults": max_results,
        "key": API_KEY
    }

    response = requests.get(
    SEARCH_URL,
    params=params,
    timeout=30
)
    response.raise_for_status()
    data = response.json()

    video_ids = []

    for item in data["items"]:
        video_id = item["id"].get("videoId")

        if video_id:
            video_ids.append(video_id)

    return video_ids

In [5]:
def get_video_details(video_ids):
    """
    Retrieve detailed information for a list of YouTube videos.

    Parameters:
        video_ids (list): List of YouTube video IDs.

    Returns:
        list: List of dictionaries containing video details.
    """

    video_ids = ",".join(video_ids)

    params = {
        "part": "snippet,statistics,contentDetails",
        "id": video_ids,
        "key": API_KEY
    }

    response = requests.get(
    VIDEOS_URL,
    params=params,
    timeout=30
)

    response.raise_for_status()

    data = response.json()
    video_details = []
    for item in data["items"]:
      video = {
        "video_id": item["id"],
          "title": item["snippet"].get("title"),
          "description": item["snippet"].get("description"),
          "published_at": item["snippet"].get("publishedAt"),
          "channel_id": item["snippet"].get("channelId"),
          "channel_title": item["snippet"].get("channelTitle"),
          "category_id": item["snippet"].get("categoryId"),
          "tags": item["snippet"].get("tags"),
          "duration": item["contentDetails"].get("duration"),
          "view_count": item["statistics"].get("viewCount"),
          "like_count": item["statistics"].get("likeCount"),
          "comment_count": item["statistics"].get("commentCount"),
          "thumbnail_url": item["snippet"].get("thumbnails", {}).get("high", {}).get("url")
      }
      video_details.append(video)

    return video_details

In [6]:
def get_channel_details(channel_ids):
    """
    Retrieve detailed information for a list of YouTube channels.

    Parameters:
        channel_ids (list): List of YouTube channel IDs.

    Returns:
        list: List of dictionaries containing channel details.
    """

    channel_ids = ",".join(channel_ids)

    params = {
        "part": "snippet,statistics",
        "id": channel_ids,
        "key": API_KEY
    }
    response = requests.get(
    CHANNELS_URL,
    params=params,
    timeout=30
)

    response.raise_for_status()
    data = response.json()
    channel_details = []

    for item in data["items"]:

        channel = {
            "channel_id": item["id"],
            "channel_title": item["snippet"].get("title"),
            "channel_description": item["snippet"].get("description"),
            "country": item["snippet"].get("country"),
            "published_at": item["snippet"].get("publishedAt"),
            "subscriber_count": item["statistics"].get("subscriberCount"),
            "video_count": item["statistics"].get("videoCount"),
            "view_count": item["statistics"].get("viewCount")
        }

        channel_details.append(channel)

    return channel_details

In [11]:
def merge_video_channel_data(videos, channels):
    """
    Merge video metadata with channel metadata.

    Parameters:
        videos (list): List of video dictionaries.
        channels (list): List of channel dictionaries.

    Returns:
        list: Combined video-channel dataset.
    """
    merged_data = []
    channel_lookup = {}

    for channel in channels:
      channel_lookup[channel["channel_id"]] = channel

    for video in videos:
      channel = channel_lookup.get(video["channel_id"], {})

      merged_record = {
          **video,
          **channel
      }

      merged_data.append(merged_record)

    return merged_data

**Collector Loop**

In [ ]:
from src.checkpoint import (
    save_checkpoint,
    load_checkpoint,
    clear_checkpoint
)

import pandas as pd
from datetime import datetime
import os
import time

master_dataset = []

# ==========================================
# CONFIGURATION
# ==========================================

DATA_DIR = "../data/raw"
CSV_PATH = os.path.join(DATA_DIR, "youtube_dataset.csv")

os.makedirs(DATA_DIR, exist_ok=True)

import pandas as pd
from pandas.errors import EmptyDataError

master_dataset = []
existing_video_ids = set()

if os.path.exists(CSV_PATH):

    try:

        existing_df = pd.read_csv(CSV_PATH)

        master_dataset = existing_df.to_dict("records")

        if "video_id" in existing_df.columns:
            existing_video_ids = set(existing_df["video_id"].astype(str))

        print(f"Loaded {len(master_dataset)} existing records.")

    except EmptyDataError:

        print("CSV exists but is empty. Starting fresh.")

        master_dataset = []
        existing_video_ids = set()

else:

    print("Starting fresh dataset.")
# ==========================================
# LOAD CHECKPOINT
# ==========================================

checkpoint = load_checkpoint()

resume = checkpoint is None

last_category = checkpoint["category"] if checkpoint else None
last_keyword = checkpoint["keyword"] if checkpoint else None

# ==========================================
# COLLECTION STARTS
# ==========================================

try:

    for category, keywords in KEYWORDS.items():

        if not resume:

            if category != last_category:
                continue

            resume = True

        print("\n" + "=" * 70)
        print(f"CATEGORY : {category}")
        print("=" * 70)

        skip_keyword = last_keyword is not None

        for keyword in keywords:

            if skip_keyword:

                if keyword != last_keyword:
                    continue

                skip_keyword = False
                continue

            print(f"\nCollecting : {keyword}")

            try:

                # ------------------------------------
                # SEARCH
                # ------------------------------------

                print("Searching videos...")

                video_ids = search_videos(
                    keyword,
                    max_results=20
                )

                if not video_ids:

                    print("No videos found.")
                    continue

                # ------------------------------------
                # VIDEO DETAILS
                # ------------------------------------

                print("Getting video details...")

                videos = get_video_details(video_ids)

                if not videos:

                    print("No video details returned.")
                    continue

                # ------------------------------------
                # CHANNEL DETAILS
                # ------------------------------------

                channel_ids = list({

                    video["channel_id"]

                    for video in videos

                    if video.get("channel_id")

                })

                print("Getting channel details...")

                channels = get_channel_details(channel_ids)

                if not channels:

                    print("No channel details returned.")
                    continue

                # ------------------------------------
                # MERGE
                # ------------------------------------

                merged_data = merge_video_channel_data(
                    videos,
                    channels
                )

                timestamp = datetime.now().strftime(
                    "%Y-%m-%d %H:%M:%S"
                )

                for row in merged_data:

                    row["search_keyword"] = keyword
                    row["search_category"] = category
                    row["collected_at"] = timestamp

                # ==========================================
                # KEEP ONLY NEW VIDEOS
                # ==========================================

                new_records = []

                for row in merged_data:

                    video_id = str(row["video_id"])

                    if video_id not in existing_video_ids:

                        existing_video_ids.add(video_id)

                        new_records.append(row)

                if new_records:

                    new_df = pd.DataFrame(new_records)

                    if os.path.exists(CSV_PATH) and os.path.getsize(CSV_PATH) > 0:

                        new_df.to_csv(
                            CSV_PATH,
                            mode="a",
                            header=False,
                            index=False
                        )

                    else:

                        new_df.to_csv(
                            CSV_PATH,
                            index=False
                        )

                    master_dataset.extend(new_records)

                print(f"Added {len(new_records)} new videos.")
                print(f"Dataset Size : {len(existing_video_ids)}")

                # ------------------------------------
                # SAVE CHECKPOINT
                # ------------------------------------

                save_checkpoint(
                    category=category,
                    keyword=keyword,
                    total_records=len(master_dataset)
                )

                print(f"Collected : {len(merged_data)} videos")
                print(f"Dataset Size : {len(master_dataset)}")

                time.sleep(1)

            except KeyboardInterrupt:

                raise

            except Exception as e:

            print(f"\nError on keyword '{keyword}'")
            print(e)

            save_checkpoint(
                category=category,
                keyword=keyword,
                total_records=len(master_dataset)
            )

            print("Progress saved.")

            raise

# ==========================================
# USER STOPPED PROGRAM
# ==========================================

except KeyboardInterrupt:

    print("\nCollection interrupted by user.")

    save_checkpoint(
        category=category,
        keyword=keyword,
        total_records=len(master_dataset)
    )

    print("Checkpoint saved.")

# ==========================================
# FINISHED
# ==========================================

else:

    clear_checkpoint()

    print("\nCollection completed successfully.")

save_checkpoint(
    category="FINISHED",
    keyword="ALL_KEYWORDS",
    total_records=len(master_dataset)
)

Starting fresh dataset.

CATEGORY : Technology

Searching videos...
Getting video details...
Getting channel details...
Added 20 new videos.
Dataset Size : 20
Collected : 20 videos
Dataset Size : 20

Searching videos...
Getting video details...
Getting channel details...
Added 20 new videos.
Dataset Size : 40
Collected : 20 videos
Dataset Size : 40

Searching videos...
Getting video details...
Getting channel details...
Added 19 new videos.
Dataset Size : 59
Collected : 19 videos
Dataset Size : 59

Searching videos...
Getting video details...
Getting channel details...
Added 20 new videos.
Dataset Size : 79
Collected : 20 videos
Dataset Size : 79

Searching videos...
Getting video details...
Getting channel details...
Added 20 new videos.
Dataset Size : 99
Collected : 20 videos
Dataset Size : 99

Searching videos...
Getting video details...
Getting channel details...
Added 20 new videos.
Dataset Size : 119
Collected : 20 videos
Dataset Size : 119

Searching videos...
Getting video deta

In [25]:
from src.checkpoint import save_checkpoint

save_checkpoint(
    category="Test",
    keyword="Python",
    total_records=2061
)

In [16]:
import os

os.makedirs("../data/raw", exist_ok=True)

In [19]:
df = pd.DataFrame(master_dataset)

if df.empty:
    print("No data collected.")
else:
    df.drop_duplicates(
        subset="video_id",
        inplace=True
    )

    os.makedirs("../data/raw", exist_ok=True)

    df.to_csv(
        "../data/raw/youtube_dataset.csv",
        index=False
    )

    print(f"Dataset saved successfully.")
    print(f"Shape: {df.shape}")

    display(df.head())

Dataset saved successfully.
Shape: (2061, 20)


,video_id,title,description,published_at,channel_id,channel_title,category_id,tags,duration,view_count,like_count,comment_count,thumbnail_url,channel_description,country,subscriber_count,video_count,search_keyword,search_category,collected_at
0,r3I1_biJ5Rs,How OpenAI technology 'went rogue' | Global Ne...,The artificial intelligence giant OpenAI - mak...,2006-04-08T05:51:05Z,UC16niRr50-MSBwiO3YDb3RA,BBC News,25,"[bbc, bbc news, news, world news, breaking new...",PT9M43S,7128283600,338,140,https://i.ytimg.com/vi/r3I1_biJ5Rs/hqdefault.jpg,"Breaking news, live updates and in-depth analy...",GB,19900000,31890,technology news,Technology,2026-07-23 13:24:57
1,rIj7DnrshxY,"Tech News 2212 || Galaxy Unpacked, vivo S2, re...","Tech News 2212 || Galaxy Unpacked, vivo S2, re...",2015-11-12T05:14:19Z,UCb-xXZ7ltTvrh9C6DgB9H-Q,Prasadtechintelugu,28,"[prasadtechintelugu, prasad, techtelugu, techn...",PT8M47S,2188859182,8255,304,https://i.ytimg.com/vi/rIj7DnrshxY/hqdefault.jpg,"Technology , Smartphone Reviews , Unboxing Gad...",IN,5140000,5351,technology news,Technology,2026-07-23 13:24:57
2,pKsEFQgpe-o,"Nvidia Rolls Out New Chips, WBD Deal In Limbo ...",Bloomberg's Ed Ludlow breaks down Nvidia's lat...,2016-09-01T21:38:32Z,UCrM7B7SL_g1edFOnmj-SDKg,Bloomberg Tech,25,"[Advisors Capital Management LLC, Bessemer Ven...",PT49M8S,177869185,139,8,https://i.ytimg.com/vi/pKsEFQgpe-o/hqdefault.jpg,Welcome to the Bloomberg Tech YouTube channel....,NaN,732000,13916,technology news,Technology,2026-07-23 13:24:57
3,qnnXBakpLoc,"Tech News 2211 || Xiaomi, Redmi Note 17, Pixel...","Tech News 2211 || Xiaomi, Redmi Note 17, Pixel...",2015-11-12T05:14:19Z,UCb-xXZ7ltTvrh9C6DgB9H-Q,Prasadtechintelugu,28,"[prasadtechintelugu, prasad, techtelugu, techn...",PT8M18S,2188859182,9297,265,https://i.ytimg.com/vi/qnnXBakpLoc/hqdefault.jpg,"Technology , Smartphone Reviews , Unboxing Gad...",IN,5140000,5351,technology news,Technology,2026-07-23 13:24:57
4,2JgJ1MRhntg,Microsoft in the race to build the ultimate su...,Computing giant Microsoft has spent 20 years p...,2006-04-08T05:51:05Z,UC16niRr50-MSBwiO3YDb3RA,BBC News,25,"[bbc, bbc news, news, world news, breaking news]",PT5M33S,7128283600,2212,341,https://i.ytimg.com/vi/2JgJ1MRhntg/hqdefault.jpg,"Breaking news, live updates and in-depth analy...",GB,19900000,31890,technology news,Technology,2026-07-23 13:24:57


In [21]:
print(len(master_dataset))

2061


In [22]:
import pandas as pd
df = pd.DataFrame(master_dataset)

In [23]:
df.to_csv("../data/raw/youtube_dataset.csv", index=False)

In [24]:
print(df.shape)
df.head()

(2061, 20)


,video_id,title,description,published_at,channel_id,channel_title,category_id,tags,duration,view_count,like_count,comment_count,thumbnail_url,channel_description,country,subscriber_count,video_count,search_keyword,search_category,collected_at
0,r3I1_biJ5Rs,How OpenAI technology 'went rogue' | Global Ne...,The artificial intelligence giant OpenAI - mak...,2006-04-08T05:51:05Z,UC16niRr50-MSBwiO3YDb3RA,BBC News,25,"[bbc, bbc news, news, world news, breaking new...",PT9M43S,7128283600,338,140,https://i.ytimg.com/vi/r3I1_biJ5Rs/hqdefault.jpg,"Breaking news, live updates and in-depth analy...",GB,19900000,31890,technology news,Technology,2026-07-23 13:24:57
1,rIj7DnrshxY,"Tech News 2212 || Galaxy Unpacked, vivo S2, re...","Tech News 2212 || Galaxy Unpacked, vivo S2, re...",2015-11-12T05:14:19Z,UCb-xXZ7ltTvrh9C6DgB9H-Q,Prasadtechintelugu,28,"[prasadtechintelugu, prasad, techtelugu, techn...",PT8M47S,2188859182,8255,304,https://i.ytimg.com/vi/rIj7DnrshxY/hqdefault.jpg,"Technology , Smartphone Reviews , Unboxing Gad...",IN,5140000,5351,technology news,Technology,2026-07-23 13:24:57
2,pKsEFQgpe-o,"Nvidia Rolls Out New Chips, WBD Deal In Limbo ...",Bloomberg's Ed Ludlow breaks down Nvidia's lat...,2016-09-01T21:38:32Z,UCrM7B7SL_g1edFOnmj-SDKg,Bloomberg Tech,25,"[Advisors Capital Management LLC, Bessemer Ven...",PT49M8S,177869185,139,8,https://i.ytimg.com/vi/pKsEFQgpe-o/hqdefault.jpg,Welcome to the Bloomberg Tech YouTube channel....,NaN,732000,13916,technology news,Technology,2026-07-23 13:24:57
3,qnnXBakpLoc,"Tech News 2211 || Xiaomi, Redmi Note 17, Pixel...","Tech News 2211 || Xiaomi, Redmi Note 17, Pixel...",2015-11-12T05:14:19Z,UCb-xXZ7ltTvrh9C6DgB9H-Q,Prasadtechintelugu,28,"[prasadtechintelugu, prasad, techtelugu, techn...",PT8M18S,2188859182,9297,265,https://i.ytimg.com/vi/qnnXBakpLoc/hqdefault.jpg,"Technology , Smartphone Reviews , Unboxing Gad...",IN,5140000,5351,technology news,Technology,2026-07-23 13:24:57
4,2JgJ1MRhntg,Microsoft in the race to build the ultimate su...,Computing giant Microsoft has spent 20 years p...,2006-04-08T05:51:05Z,UC16niRr50-MSBwiO3YDb3RA,BBC News,25,"[bbc, bbc news, news, world news, breaking news]",PT5M33S,7128283600,2212,341,https://i.ytimg.com/vi/2JgJ1MRhntg/hqdefault.jpg,"Breaking news, live updates and in-depth analy...",GB,19900000,31890,technology news,Technology,2026-07-23 13:24:57
